### Summerization


In [2]:
import os

from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

In [7]:
agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep = ("messages",4)
        )
        ]

)

In [8]:
config={"configurable": {"thread_id": "1"}}

In [9]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

In [ ]:
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='fda99305-4e36-46ee-a923-b1f8e7b9e29b'), AIMessage(content='<think>\nOkay, so the user is asking "What is 2+2?" Hmm, that seems straightforward, but maybe I should make sure I\'m not missing anything. Let me start by recalling basic arithmetic. Addition is one of the fundamental operations in mathematics. When you add two numbers together, you combine their quantities.\n\nSo, 2 plus 2... Well, in the decimal system we commonly use, adding two and two gives four. But wait, is there any context where this might not be the case? For example, in different number bases? Let me think. In base 10, which is standard, 2+2 is 4. If we were in base 3, the digits go 0,1,2, and then 10. So 2+2 in base 3 would be 11, which is 4 in decimal. But the question doesn\'t specify a different base, so it\'s safe to assume base 10.\n\nAnother angle: maybe in some abstract algebra or modulo arithmetic? 

In [12]:
response["messages"]

[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nAlright, let\'s tackle this. The user\'s latest question is "What is 15-7?" Looking at the conversation history, they\'ve been asking simple arithmetic problems one after another: 2+2, 10*5, 100/4, and now 15-7. Each time, the assistant has provided a straightforward answer with a brief explanation and a boxed result.\n\nFirst, I need to determine the primary goal or session intent. The user seems to be testing basic arithmetic skills or practicing math problems. The assistant\'s role here is to answer each question accurately and consistently.\n\nNext, the summary should capture the key points from the conversation. The user\'s pattern is clear: they\'re asking for simple operations, and the assistant is responding with correct answers. Each step involves a different operation, so the summary needs to note each problem and its solution without unnecessary details.\n\nArtifacts in this context would be th

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.tools import tool

In [4]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

In [6]:
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer = InMemorySaver(),
    tools=[search_hotels],
    middleware=[
    SummarizationMiddleware(
        model="groq:qwen/qwen3-32b",
        trigger = ("tokens",550),
        keep = ("tokens",200)
    )
    ]
)

In [8]:
config = {"configurable": {"thread_id": "1"}}

In [ ]:
def token_counter(messages):
    total_chars = sum(len(m.content) for m in messages)
    return total_chars//4  # Rough estimate: 1 token ~ 4 characters

In [11]:
# cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

# for city in cities:
#     response = agent.invoke({"messages":[HumanMessage(content=f"Search hotels in {city}")]},config)
#     tokens = token_counter(response["messages"])
#     print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
#     print(f"{(response['messages'])}")

In [55]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import SystemMessage, HumanMessage

from langgraph.checkpoint.memory import InMemorySaver


In [56]:
def read_email(email_id: str) -> str:
    """Read email - returns long response to use more tokens."""
    return f"Email content for id {email_id}:"

def send_email(recipent: str, subject: str, body: str) -> str:
    """mock function to send to an email."""
    return f"Email sent to {recipent} with subject {subject}"
    

In [57]:
agent = create_agent(
    model= "groq:qwen/qwen3-32b",
    tools=[read_email, send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on ={
                "send_email":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email":False,
            }
        )
    ]
)

In [58]:
config = {"configurable": {"thread_id": "test-reject"}}

In [59]:
result = agent.invoke({"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},config)

In [60]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='04e09020-b072-40cc-a0e9-c60e8a04b766'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email function. The parameters required are recipent, subject, and body. The user provided all three: recipent is john@test.com, subject is Hello, and body is How are you? So I need to call send_email with these arguments. I should make sure the JSON is correctly formatted with the right parameter names. Let me double-check the spelling of the parameters. Recipent is spelled correctly, subject and body too. Alright, that's all. No other functions needed here.\n", 'tool_calls': [{'id': 'cee9916nj', 'function': {'arguments': '{"body":"How are you?","recipent":"john@test.com","s

In [61]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(Command(resume={"decisions":[{"type":"reject"}]}),config)

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It seems that sending the email to john@test.com with the subject "Hello" could not be completed at this time. Please verify the recipient details or try again later. Let me know how you'd like to proceed!


In [62]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='04e09020-b072-40cc-a0e9-c60e8a04b766'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email function. The parameters required are recipent, subject, and body. The user provided all three: recipent is john@test.com, subject is Hello, and body is How are you? So I need to call send_email with these arguments. I should make sure the JSON is correctly formatted with the right parameter names. Let me double-check the spelling of the parameters. Recipent is spelled correctly, subject and body too. Alright, that's all. No other functions needed here.\n", 'tool_calls': [{'id': 'cee9916nj', 'function': {'arguments': '{"body":"How are you?","recipent":"john@test.com","s